# DINO+SAM2 2.5D Tracking Feasibility Demo

This notebook tests whether existing DINO+SAM2 object masks plus monocular depth can provide useful pseudo-3D anchors for preserving object identity through missed detections, occlusion, and label flips.

It does **not** run YOLO. It does **not** rerun DINO/SAM2. It does **not** implement SLAM, NeRF, 3DGS, DUSt3R, MASt3R, or full static reconstruction.

## 1. Setup and Imports

In [ ]:
from __future__ import annotations

import ast
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    display = print

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    DEVICE = "cpu"

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    return start

REPO_ROOT = find_repo_root()
print("repo root:", REPO_ROOT)
print("device:", DEVICE)

## 2. Config

In [ ]:
CONFIG = {
    "VIDEO_PATH": "",
    "FRAMES_DIR": "data/auto_label_demo/frames",
    "DINO_SAM2_RESULTS_PATH": "data/auto_label_demo/embeddings/object_metadata.parquet",
    "PROPOSALS_DIR": "data/auto_label_demo/proposals",
    "USE_PROPOSAL_ID_ASSET_LOOKUP": True,
    "CROPS_DIR": "",
    "OUTPUT_DIR": "outputs/demo_dino_sam2_2p5d_tracking_feasibility",
    "DEPTH_MODEL": "dummy",  # "depth_anything_v2", "midas", or "dummy"
    "DEPTH_INVERT": False,
    "MAX_FRAMES": 100,
    "MIN_MASK_AREA": 80,
    "MIN_DEPTH_PIXELS": 40,
    "MATCH_THRESHOLD": 0.35,
    "IOU_WEIGHT": 0.35,
    "CENTER_WEIGHT": 0.20,
    "DEPTH_WEIGHT": 0.25,
    "LABEL_WEIGHT": 0.10,
    "CROP_APPEARANCE_WEIGHT": 0.10,
    "MAX_MISSING_FRAMES": 15,
    "MAX_OCCLUDED_FRAMES": 60,
    "EMA_ALPHA": 0.7,
    "OCCLUSION_OVERLAP": 0.15,
    "OCCLUDER_CLOSER_MARGIN": 0.03,
    "SAVE_DEBUG_JSON": True,
    "SAVE_VIS_VIDEO": True,
}

OUTPUT_DIR = REPO_ROOT / CONFIG["OUTPUT_DIR"]
FRAME_OUT_DIR = OUTPUT_DIR / "frames"
VIS_DIR = OUTPUT_DIR / "selected_visualizations"
for p in [OUTPUT_DIR, FRAME_OUT_DIR, VIS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))

## 3. Data Loading

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def repo_path(value: str | Path) -> Path:
    p = Path(value)
    if p.is_absolute() or p.exists():
        return p
    return REPO_ROOT / p

def parse_maybe_list(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (list, tuple, np.ndarray)):
        return [float(x) for x in value[:4]]
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            out = parser(text)
            if isinstance(out, (list, tuple)) and len(out) >= 4:
                return [float(x) for x in out[:4]]
        except Exception:
            pass
    return None

def resolve_existing(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    candidates = [Path(text)]
    if not Path(text).is_absolute():
        candidates.extend([REPO_ROOT / text, REPO_ROOT / "notebooks" / text])
    for p in candidates:
        if p.exists():
            return p
    return None

def list_frames(frames_dir: Path) -> list[Path]:
    if not frames_dir.exists():
        return []
    return sorted(p for p in frames_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

def extract_video_frames(video_path: Path, out_dir: Path, max_frames: int | None = None) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    rows = []
    idx = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            out = out_dir / f"frame_{idx:06d}.jpg"
            cv2.imwrite(str(out), frame)
            rows.append(out)
            idx += 1
            if max_frames and idx >= max_frames:
                break
    finally:
        cap.release()
    return rows

def load_result_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".jsonl":
        return pd.DataFrame(json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip())
    if path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict) and "frames" in data:
            rows = []
            for fr in data["frames"]:
                for det in fr.get("detections", []):
                    rows.append({**det, "frame_idx": fr.get("frame_idx"), "frame_path": fr.get("frame_path")})
            return pd.DataFrame(rows)
        return pd.DataFrame(data.get("detections", []))
    raise ValueError(f"Unsupported result format: {path}")

frames_dir = repo_path(CONFIG["FRAMES_DIR"]) if CONFIG["FRAMES_DIR"] else Path("")
video_path = repo_path(CONFIG["VIDEO_PATH"]) if CONFIG["VIDEO_PATH"] else Path("")
frames = list_frames(frames_dir)
if not frames and video_path.exists():
    frames = extract_video_frames(video_path, FRAME_OUT_DIR, CONFIG["MAX_FRAMES"])
if CONFIG["MAX_FRAMES"]:
    frames = frames[: int(CONFIG["MAX_FRAMES"])]
frame_by_name = {p.name: p for p in frames}
frame_by_stem = {p.stem: p for p in frames}
print("frames loaded:", len(frames))

results_path = repo_path(CONFIG["DINO_SAM2_RESULTS_PATH"])
raw_df = load_result_table(results_path)
print("raw results shape:", raw_df.shape)
print("raw columns:", list(raw_df.columns))

def infer_frame_idx_from_path(path: Path | None, fallback: int) -> int:
    if path is None:
        return fallback
    digits = "".join(ch if ch.isdigit() else " " for ch in path.stem).split()
    return int(digits[-1]) if digits else fallback

def clean_int(value, fallback: int | None = None) -> int | None:
    try:
        if value is None or (isinstance(value, float) and math.isnan(value)) or pd.isna(value):
            return fallback
    except Exception:
        pass
    try:
        return int(value)
    except Exception:
        return fallback

def proposal_asset_path(proposal_id, kind: str) -> Path | None:
    if not CONFIG.get("USE_PROPOSAL_ID_ASSET_LOOKUP", True):
        return None
    pid = clean_int(proposal_id)
    if pid is None:
        return None
    root = repo_path(CONFIG.get("PROPOSALS_DIR", ""))
    if kind == "mask":
        p = root / "masks" / f"mask_{pid:07d}.png"
    elif kind == "crop":
        p = root / "crops" / f"proposal_{pid:07d}.jpg"
    else:
        return None
    return p if p.exists() else None

def normalize_detection_row(row, idx: int) -> dict[str, Any]:
    frame_path = resolve_existing(row.get("frame_path") or row.get("image_path"))
    bbox = parse_maybe_list(row.get("bbox") if "bbox" in row else row.get("bbox_xyxy"))
    if bbox is None and all(c in row for c in ["x1", "y1", "x2", "y2"]):
        bbox = [float(row["x1"]), float(row["y1"]), float(row["x2"]), float(row["y2"])]
    label = str(row.get("label") or row.get("class_name") or row.get("dino_label") or row.get("cluster_label") or row.get("predicted_label") or "unknown")
    conf = float(row.get("conf") or row.get("confidence") or row.get("score") or row.get("dino_score") or 0.0)
    proposal_id = clean_int(row.get("proposal_id"), None)
    det_id = proposal_id if proposal_id is not None else clean_int(row.get("embedding_idx"), idx)
    mask_path = resolve_existing(row.get("mask_path")) if "mask_path" in row else None
    crop_path = resolve_existing(row.get("crop_path")) if "crop_path" in row else None
    if mask_path is None:
        mask_path = proposal_asset_path(proposal_id, "mask")
    if crop_path is None:
        crop_path = proposal_asset_path(proposal_id, "crop")
    return {
        "det_id": int(det_id),
        "proposal_id": proposal_id,
        "frame_idx": clean_int(row.get("frame_idx"), None) if "frame_idx" in row else infer_frame_idx_from_path(frame_path, idx),
        "frame_path": frame_path,
        "label": label,
        "conf": conf,
        "bbox": bbox,
        "mask_path": mask_path,
        "crop_path": crop_path,
        "cluster_label": str(row.get("cluster_label") or row.get("cluster_suggested_label") or row.get("suggested_label") or ""),
        "raw_metadata": row.to_dict(),
    }

all_dets = [normalize_detection_row(row, i) for i, (_, row) in enumerate(raw_df.iterrows())]

# Align detections to the actual frame list by full resolved path.
# Avoid grouping only by names like frame_0000000.jpg, because many videos share the same basename.
def resolved_key(path: Path | None) -> str | None:
    if path is None:
        return None
    try:
        return str(path.resolve())
    except Exception:
        return str(path)

valid_frame_set = {resolved_key(p) for p in frames}
valid_frame_set.discard(None)
if valid_frame_set:
    matched_dets = [d for d in all_dets if resolved_key(d.get("frame_path")) in valid_frame_set]
    if matched_dets:
        all_dets = matched_dets
    else:
        metadata_frames = []
        seen = set()
        for d in all_dets:
            fp = d.get("frame_path")
            key = resolved_key(fp)
            if fp is not None and key not in seen and Path(fp).exists():
                metadata_frames.append(Path(fp))
                seen.add(key)
            if CONFIG["MAX_FRAMES"] and len(metadata_frames) >= int(CONFIG["MAX_FRAMES"]):
                break
        if metadata_frames:
            print("Warning: loaded FRAMES_DIR did not match metadata frame_path values; using frames referenced by DINO+SAM2 metadata instead.")
            frames = metadata_frames
            valid_frame_set = {resolved_key(p) for p in frames}
            all_dets = [d for d in all_dets if resolved_key(d.get("frame_path")) in valid_frame_set]
        else:
            print("Warning: no exact frame_path match and no metadata frames could be opened; detections may be empty.")
            all_dets = []

frame_index_by_path = {resolved_key(p): i for i, p in enumerate(frames)}
dets_by_frame = defaultdict(list)
for d in all_dets:
    key = resolved_key(d.get("frame_path"))
    if key not in frame_index_by_path:
        continue
    d["frame_idx"] = frame_index_by_path[key]
    dets_by_frame[d["frame_idx"]].append(d)

summary = {
    "frames_loaded": len(frames),
    "detections": len(all_dets),
    "masks_from_file": sum(1 for d in all_dets if d.get("mask_path")),
    "crops_from_file": sum(1 for d in all_dets if d.get("crop_path")),
    "bbox_fallback_masks": sum(1 for d in all_dets if not d.get("mask_path") and d.get("bbox")),
    "labels": Counter(d["label"] for d in all_dets).most_common(20),
}
display(pd.DataFrame([summary]))
print("top labels:", summary["labels"])

## 4. Visualization of Raw Inputs

In [ ]:
def read_rgb(path: Path):
    img = cv2.imread(str(path))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

def load_mask_for_det(det, shape_hw):
    h, w = shape_hw
    if det.get("mask_path") and Path(det["mask_path"]).exists():
        m = cv2.imread(str(det["mask_path"]), cv2.IMREAD_GRAYSCALE)
        if m is not None:
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            return m > 0, "file"
    bbox = det.get("bbox")
    mask = np.zeros((h, w), dtype=bool)
    if bbox is not None:
        x1, y1, x2, y2 = [int(round(v)) for v in bbox]
        x1, y1, x2, y2 = max(0,x1), max(0,y1), min(w,x2), min(h,y2)
        if x2 > x1 and y2 > y1:
            mask[y1:y2, x1:x2] = True
    return mask, "bbox_fallback"

def overlay_dets(img, dets, alpha=0.35):
    out = img.copy()
    rng = np.random.default_rng(123)
    for det in dets:
        mask, _ = load_mask_for_det(det, img.shape[:2])
        color = rng.integers(40, 255, size=3, dtype=np.uint8)
        out[mask] = (out[mask] * (1-alpha) + color * alpha).astype(np.uint8)
        if det.get("bbox"):
            x1,y1,x2,y2 = [int(round(v)) for v in det["bbox"]]
            cv2.rectangle(out, (x1,y1), (x2,y2), color.tolist(), 2)
            cv2.putText(out, str(det["label"])[:24], (x1, max(15, y1-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color.tolist(), 1)
    return out

sample_frame_paths = frames[:5]
fig, axes = plt.subplots(len(sample_frame_paths), 2, figsize=(12, 4*len(sample_frame_paths)))
if len(sample_frame_paths) == 1:
    axes = np.array([axes])
for r, path in enumerate(sample_frame_paths):
    img = read_rgb(path)
    if img is None:
        continue
    frame_idx = r
    dets = dets_by_frame.get(frame_idx, [])
    axes[r,0].imshow(img); axes[r,0].set_title(path.name); axes[r,0].axis("off")
    axes[r,1].imshow(overlay_dets(img, dets)); axes[r,1].set_title(f"DINO+SAM2 dets: {len(dets)}"); axes[r,1].axis("off")
plt.tight_layout(); plt.show()

crop_paths = [d["crop_path"] for d in all_dets if d.get("crop_path")][:12]
if crop_paths:
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    for ax, p in zip(axes.flat, crop_paths):
        im = read_rgb(p)
        if im is not None:
            ax.imshow(im)
        ax.set_title(Path(p).name[:18]); ax.axis("off")
    plt.tight_layout(); plt.show()

## 5. Depth Model Wrapper

In [ ]:
class DepthWrapper:
    def __init__(self, model_name: str, device: str, invert: bool = False):
        self.model_name = model_name
        self.device = device
        self.invert = invert
        self.model = None
        self.transform = None
        self.actual = "dummy"
        self._load()

    def _load(self):
        if self.model_name == "depth_anything_v2":
            try:
                # Placeholder: install/load your local Depth Anything V2 here.
                # Example future hook:
                # from depth_anything_v2.dpt import DepthAnythingV2
                raise ImportError("Depth Anything V2 loader not configured in this demo")
            except Exception as exc:
                print("Depth Anything V2 unavailable; falling back to MiDaS/dummy:", exc)
                self.model_name = "midas"
        if self.model_name == "midas" and torch is not None:
            try:
                self.model = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)
                self.model.to(self.device).eval()
                transforms = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True)
                self.transform = transforms.small_transform
                self.actual = "midas"
                print("MiDaS loaded")
                return
            except Exception as exc:
                print("MiDaS unavailable; using dummy depth:", exc)
        self.actual = "dummy"
        print("Using dummy depth")

    def predict(self, rgb: np.ndarray) -> np.ndarray:
        h, w = rgb.shape[:2]
        if self.actual == "midas":
            inp = self.transform(rgb).to(self.device)
            with torch.inference_mode():
                pred = self.model(inp)
                pred = torch.nn.functional.interpolate(pred.unsqueeze(1), size=(h, w), mode="bicubic", align_corners=False).squeeze()
            depth = pred.detach().float().cpu().numpy()
        else:
            y = np.linspace(0, 1, h, dtype=np.float32)[:, None]
            gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
            depth = 0.65 * (1.0 - y) + 0.35 * gray
        depth = depth.astype(np.float32)
        depth -= np.nanmin(depth)
        depth /= max(1e-6, np.nanmax(depth))
        if self.invert:
            depth = 1.0 - depth
        return depth

depth_model = DepthWrapper(CONFIG["DEPTH_MODEL"], DEVICE, CONFIG["DEPTH_INVERT"])

if frames:
    img = read_rgb(frames[0])
    d = depth_model.predict(img)
    fig, ax = plt.subplots(1,2,figsize=(10,4))
    ax[0].imshow(img); ax[0].set_title("RGB"); ax[0].axis("off")
    ax[1].imshow(d, cmap="magma"); ax[1].set_title(f"depth: {depth_model.actual}"); ax[1].axis("off")
    plt.tight_layout(); plt.show()

## 6. Detection Enrichment with 2.5D Anchors

In [ ]:
def crop_from_bbox(rgb, bbox):
    h, w = rgb.shape[:2]
    x1,y1,x2,y2 = [int(round(v)) for v in bbox]
    x1,y1,x2,y2 = max(0,x1), max(0,y1), min(w,x2), min(h,y2)
    return rgb[y1:y2, x1:x2] if x2 > x1 and y2 > y1 else None

def color_hist_embedding(crop):
    if crop is None or crop.size == 0:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0,1,2], None, [8,4,4], [0,180,0,256,0,256]).flatten().astype(np.float32)
    hist /= max(1e-6, np.linalg.norm(hist))
    return hist

def enrich_detection(det, rgb, depth):
    h, w = rgb.shape[:2]
    bbox = det.get("bbox")
    if bbox is None:
        return None
    mask, mask_source = load_mask_for_det(det, (h, w))
    area = int(mask.sum())
    if area < CONFIG["MIN_MASK_AREA"]:
        return None
    vals = depth[mask]
    vals = vals[np.isfinite(vals)]
    if len(vals) < CONFIG["MIN_DEPTH_PIXELS"]:
        return None
    x1,y1,x2,y2 = [float(v) for v in bbox]
    cx, cy = (x1+x2)/2, (y1+y2)/2
    p20, med, p80 = np.percentile(vals, [20,50,80])
    crop = read_rgb(det["crop_path"]) if det.get("crop_path") else crop_from_bbox(rgb, bbox)
    app = color_hist_embedding(crop)
    return {
        **det,
        "mask": mask,
        "mask_source": mask_source,
        "area": area,
        "center_2d": [cx, cy],
        "median_depth": float(med),
        "depth_p20": float(p20),
        "depth_p80": float(p80),
        "depth_std": float(np.std(vals)),
        "pseudo_3d_center": [float(cx/w), float(cy/h), float(med)],
        "pseudo_3d_extent": [float((x2-x1)/w), float((y2-y1)/h), float(p80-p20)],
        "appearance": app,
    }

enriched_by_frame = {}
depth_by_frame = {}
for i, path in enumerate(tqdm(frames, desc="depth + enrich")):
    rgb = read_rgb(path)
    if rgb is None:
        continue
    frame_idx = i
    depth = depth_model.predict(rgb)
    depth_by_frame[frame_idx] = depth
    rows = []
    for det in dets_by_frame.get(frame_idx, []):
        enr = enrich_detection(det, rgb, depth)
        if enr is not None:
            rows.append(enr)
    enriched_by_frame[frame_idx] = rows
print("enriched detections:", sum(len(v) for v in enriched_by_frame.values()))

## 7-8. Baseline 2D Tracker and 2.5D Tracker

In [ ]:
def bbox_iou(a,b):
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    ix1,iy1,ix2,iy2 = max(ax1,bx1), max(ay1,by1), min(ax2,bx2), min(ay2,by2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter <= 0: return 0.0
    aa = max(0, ax2-ax1) * max(0, ay2-ay1); ba = max(0, bx2-bx1) * max(0, by2-by1)
    return inter / max(1e-6, aa + ba - inter)

def center_score(a, b, wh):
    ax,ay = a; bx,by = b; w,h = wh
    dist = math.sqrt(((ax-bx)/max(1,w))**2 + ((ay-by)/max(1,h))**2)
    return float(math.exp(-6.0 * dist))

def depth_score(a,b):
    return float(math.exp(-8.0 * abs(float(a)-float(b))))

def cosine(a,b, default=0.5):
    if a is None or b is None: return default
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a,b) / den) if den > 0 else default

class GreedyTracker:
    def __init__(self, use_25d: bool):
        self.use_25d = use_25d
        self.next_id = 1
        self.active = []
        self.finished = []
        self.history = defaultdict(list)
        self.recovered = 0

    def match_score(self, tr, det, wh):
        iou = bbox_iou(tr["bbox"], det["bbox"])
        cscore = center_score(tr["center_2d"], det["center_2d"], wh)
        lscore = 1.0 if str(tr["label"]).lower() == str(det["label"]).lower() else 0.3
        if not self.use_25d:
            return 0.55*iou + 0.30*cscore + 0.15*lscore
        dscore = depth_score(tr["depth_median"], det["median_depth"])
        ascore = cosine(tr.get("appearance"), det.get("appearance"))
        return (
            CONFIG["IOU_WEIGHT"] * iou
            + CONFIG["CENTER_WEIGHT"] * cscore
            + CONFIG["DEPTH_WEIGHT"] * dscore
            + CONFIG["LABEL_WEIGHT"] * lscore
            + CONFIG["CROP_APPEARANCE_WEIGHT"] * ascore
        )

    def create_track(self, det, frame_idx):
        tr = {
            "track_id": self.next_id,
            "label": det["label"],
            "label_history": [det["label"]],
            "bbox": det["bbox"],
            "center_2d": det["center_2d"],
            "center_3d": det["pseudo_3d_center"],
            "depth_median": det["median_depth"],
            "depth_range": [det["depth_p20"], det["depth_p80"]],
            "appearance": det.get("appearance"),
            "age": 1,
            "missed": 0,
            "status": "visible",
            "last_seen_frame": frame_idx,
            "mask_source": det.get("mask_source", ""),
            "status_history": ["visible"],
            "depth_history": [det["median_depth"]],
            "frames": [frame_idx],
        }
        self.next_id += 1
        self.active.append(tr)
        return tr

    def update_track(self, tr, det, frame_idx):
        if tr["status"] in {"missing", "occluded"} and tr["missed"] > 0:
            self.recovered += 1
        a = CONFIG["EMA_ALPHA"]
        tr["bbox"] = det["bbox"]
        tr["center_2d"] = det["center_2d"]
        tr["center_3d"] = (a*np.array(tr["center_3d"]) + (1-a)*np.array(det["pseudo_3d_center"])).tolist()
        tr["depth_median"] = float(a*tr["depth_median"] + (1-a)*det["median_depth"])
        tr["depth_range"] = [det["depth_p20"], det["depth_p80"]]
        tr["appearance"] = det.get("appearance") if tr.get("appearance") is None else (a*tr["appearance"] + (1-a)*det.get("appearance", tr["appearance"]))
        tr["label_history"].append(det["label"])
        tr["label"] = Counter(tr["label_history"]).most_common(1)[0][0]
        tr["age"] += 1; tr["missed"] = 0; tr["status"] = "visible"; tr["last_seen_frame"] = frame_idx
        tr["status_history"].append("visible"); tr["depth_history"].append(det["median_depth"]); tr["frames"].append(frame_idx)

    def mark_unmatched(self, tr, visible_dets):
        occluded = False
        for det in visible_dets:
            if bbox_iou(tr["bbox"], det["bbox"]) > CONFIG["OCCLUSION_OVERLAP"] and det["median_depth"] < tr["depth_median"] - CONFIG["OCCLUDER_CLOSER_MARGIN"]:
                occluded = True
                break
        tr["missed"] += 1
        tr["status"] = "occluded" if occluded else "missing"
        tr["status_history"].append(tr["status"])
        tr["depth_history"].append(tr["depth_median"])

    def step(self, frame_idx, dets, image_shape):
        h,w = image_shape[:2]
        pairs = []
        for ti,tr in enumerate(self.active):
            for di,det in enumerate(dets):
                pairs.append((self.match_score(tr, det, (w,h)), ti, di))
        pairs.sort(reverse=True)
        matched_t, matched_d = set(), set()
        for score, ti, di in pairs:
            if score < CONFIG["MATCH_THRESHOLD"] or ti in matched_t or di in matched_d:
                continue
            self.update_track(self.active[ti], dets[di], frame_idx)
            matched_t.add(ti); matched_d.add(di)
        for ti,tr in enumerate(list(self.active)):
            if ti not in matched_t:
                self.mark_unmatched(tr, [dets[i] for i in matched_d])
        for di,det in enumerate(dets):
            if di not in matched_d:
                self.create_track(det, frame_idx)
        keep = []
        for tr in self.active:
            limit = CONFIG["MAX_OCCLUDED_FRAMES"] if tr["status"] == "occluded" else CONFIG["MAX_MISSING_FRAMES"]
            if tr["missed"] > limit:
                self.finished.append(tr)
            else:
                keep.append(tr)
                self.history[frame_idx].append(dict(tr))
        self.active = keep

    def all_tracks(self):
        return self.finished + self.active

def run_tracker(use_25d: bool):
    tracker = GreedyTracker(use_25d)
    for i,path in enumerate(tqdm(frames, desc="2.5D tracker" if use_25d else "2D tracker")):
        rgb = read_rgb(path)
        if rgb is None: continue
        frame_idx = i
        tracker.step(frame_idx, enriched_by_frame.get(frame_idx, []), rgb.shape)
    tracker.finished.extend(tracker.active)
    return tracker

tracker_2d = run_tracker(False)
tracker_25d = run_tracker(True)
print("2D tracks:", len(tracker_2d.all_tracks()), "2.5D tracks:", len(tracker_25d.all_tracks()))

## 9. Comparison and Evaluation

In [ ]:
def unique_tracks(tracker):
    by_id = {}
    for t in tracker.all_tracks():
        tid = t.get("track_id")
        if tid not in by_id or len(t.get("frames", [])) > len(by_id[tid].get("frames", [])):
            by_id[tid] = t
    return list(by_id.values())

def eval_tracker(tracker, name):
    tracks = unique_tracks(tracker)
    lengths = [len(set(t.get("frames", []))) for t in tracks]
    flips = 0
    for t in tracks:
        hist = t.get("label_history", [])
        flips += sum(1 for a,b in zip(hist, hist[1:]) if a != b)
    occluded_frames = sum(1 for frame, trs in tracker.history.items() if any(t.get("status") == "occluded" for t in trs))
    # Rough fragmentation proxy: many short tracks with same label.
    short = sum(1 for l in lengths if l < 3)
    frag = short
    return {
        "method": name,
        "tracks_created": len(tracks),
        "avg_track_length": float(np.mean(lengths)) if lengths else 0,
        "short_tracks": short,
        "label_flips": flips,
        "recovered_tracks": tracker.recovered,
        "estimated_fragmentations": frag,
        "frames_with_occluded_track_preserved": occluded_frames,
        "pct_frames_occluded_preserved": occluded_frames / max(1, len(frames)),
    }

summary_df = pd.DataFrame([eval_tracker(tracker_2d, "baseline_2d"), eval_tracker(tracker_25d, "tracker_2p5d")])
display(summary_df)

def track_examples(tracker, n=8):
    rows = []
    for t in sorted(unique_tracks(tracker), key=lambda x: -len(x.get("frames", [])))[:n]:
        rows.append({
            "track_id": t["track_id"],
            "label": t["label"],
            "length": len(t.get("frames", [])),
            "label_history": t.get("label_history", [])[:12],
            "depth_history": [round(float(x),3) for x in t.get("depth_history", [])[:12]],
            "status_history": t.get("status_history", [])[:12],
        })
    return pd.DataFrame(rows)
display(track_examples(tracker_25d))

## 10. Visualization of Tracking Result

In [ ]:
def draw_tracks(rgb, depth, frame_idx, tracker_history, title):
    canvas = rgb.copy()
    trs = tracker_history.get(frame_idx, [])
    for tr in trs:
        x1,y1,x2,y2 = [int(round(v)) for v in tr["bbox"]]
        color = (0,255,0) if tr["status"] == "visible" else (255,180,0) if tr["status"] == "occluded" else (160,160,160)
        cv2.rectangle(canvas, (x1,y1), (x2,y2), color, 2 if tr["status"] == "visible" else 1)
        text = f"id{tr['track_id']} {tr['label']} d={tr['depth_median']:.2f} {tr['status']}"
        cv2.putText(canvas, text, (x1, max(16,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return canvas

selected = frames[::max(1, len(frames)//8)][:8]
vis_paths = []
for i,path in enumerate(selected):
    rgb = read_rgb(path)
    if rgb is None: continue
    frame_idx = i
    depth = depth_by_frame.get(frame_idx)
    if depth is None: depth = depth_model.predict(rgb)
    img2d = draw_tracks(rgb, depth, frame_idx, tracker_2d.history, "2D")
    img25 = draw_tracks(rgb, depth, frame_idx, tracker_25d.history, "2.5D")
    fig, ax = plt.subplots(1,3,figsize=(16,5))
    ax[0].imshow(img2d); ax[0].set_title("baseline 2D"); ax[0].axis("off")
    ax[1].imshow(img25); ax[1].set_title("2.5D anchors"); ax[1].axis("off")
    ax[2].imshow(depth, cmap="magma"); ax[2].set_title("depth"); ax[2].axis("off")
    out = VIS_DIR / f"tracking_{frame_idx:06d}.png"
    fig.tight_layout(); fig.savefig(out, dpi=140); plt.close(fig)
    vis_paths.append(out)
    display(plt.imread(out))
print("saved visualizations:", len(vis_paths))

## 11. Save Output

In [ ]:
def serialize_tracker(tracker):
    by_id = {}
    for t in tracker.all_tracks():
        tid = t.get("track_id")
        if tid in by_id and len(t.get("frames", [])) <= len(by_id[tid].get("frames", [])):
            continue
        clean = {}
        for k,v in t.items():
            if k == "appearance":
                continue
            clean[k] = v
        by_id[tid] = clean
    return list(by_id.values())

if CONFIG["SAVE_DEBUG_JSON"]:
    (OUTPUT_DIR / "tracking_debug_2d.json").write_text(json.dumps(serialize_tracker(tracker_2d), indent=2, default=str), encoding="utf-8")
    (OUTPUT_DIR / "tracking_debug_2p5d.json").write_text(json.dumps(serialize_tracker(tracker_25d), indent=2, default=str), encoding="utf-8")
summary_df.to_csv(OUTPUT_DIR / "tracking_summary.csv", index=False)
run_log = {
    "config": CONFIG,
    "frames_loaded": len(frames),
    "detections_loaded": len(all_dets),
    "enriched_detections": sum(len(v) for v in enriched_by_frame.values()),
    "masks_from_file": sum(1 for d in all_dets if d.get("mask_path")),
    "bbox_fallback_masks": sum(1 for d in all_dets if not d.get("mask_path") and d.get("bbox")),
    "summary": summary_df.to_dict(orient="records"),
}
(OUTPUT_DIR / "run_log.json").write_text(json.dumps(run_log, indent=2, default=str), encoding="utf-8")

if CONFIG["SAVE_VIS_VIDEO"] and vis_paths:
    first = cv2.imread(str(vis_paths[0]))
    h,w = first.shape[:2]
    out_video = OUTPUT_DIR / "demo_2p5d_tracking_result.mp4"
    writer = cv2.VideoWriter(str(out_video), cv2.VideoWriter_fourcc(*"mp4v"), 4, (w,h))
    for p in vis_paths:
        frame = cv2.imread(str(p))
        if frame is not None:
            writer.write(frame)
    writer.release()
    print("video:", out_video)
print("output dir:", OUTPUT_DIR)

## 12. Feasibility Decision Cell

In [ ]:
base = summary_df[summary_df["method"] == "baseline_2d"].iloc[0]
trk = summary_df[summary_df["method"] == "tracker_2p5d"].iloc[0]
if trk["estimated_fragmentations"] < base["estimated_fragmentations"] and trk["frames_with_occluded_track_preserved"] > 0:
    verdict = "Feasibility: promising"
elif trk["estimated_fragmentations"] <= base["estimated_fragmentations"] or trk["recovered_tracks"] >= base["recovered_tracks"]:
    verdict = "Feasibility: uncertain, inspect visualizations"
else:
    verdict = "Feasibility: weak, depth or mask quality may be insufficient"
print(verdict)
print()
print("Diagnostics:")
print("- If depth values are unstable: try DEPTH_INVERT, use a better depth model, or smooth depth over time.")
print("- If masks are noisy: increase MIN_MASK_AREA or use only high-confidence DINO+SAM2 detections.")
print("- If label flips dominate: increase appearance weight and reduce label weight.")
print("- If many false occlusions happen: increase occlusion overlap threshold or require occluder to be closer by a margin.")
print("- If re-identification fails: add CLIP/DINOv2 embeddings from crops.")

## 13. Important Design Notes

- This demo is 2.5D, not full 3D.
- It tests whether object masks + monocular depth can form useful tracking anchors.
- Static rough 3D map integration can be added later by replacing `pseudo_3d_center` with global scene coordinates from MASt3R/DUSt3R/SLAM.
- Dynamic objects should be kept in a separate dynamic layer and should not pollute the static scene map.
- The current crop histogram appearance feature is only a placeholder; CLIP/DINOv2 crop embeddings can improve re-identification.
- Hungarian matching could replace the greedy matching block when the basic signal is validated.